# Heart Disease UCI Dataset - Exploratory Data Analysis (EDA)
This notebook performs comprehensive exploratory data analysis on the Cleveland Heart Disease dataset from the UCI Machine Learning Repository.

### Objective:
1. Understand the distribution of characteristics of patients.
2. Identify missing values and determine handling strategy.
3. Perform feature relationship analysis.
4. Visualize features to guide preprocessing and modeling.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Define column names
COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

## 1. Loading the Dataset
We read the raw dataset from `data/raw` and replace the placeholder `?` indicating missing values with `NaN`.

In [ ]:
data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "raw", "processed.cleveland.data")

if not os.path.exists(data_path):
    # Fallback to local relative path if executing directly
    data_path = "../data/raw/processed.cleveland.data"

df = pd.read_csv(data_path, header=None, names=COLUMN_NAMES)
# Replace missing values represented as '?' with NaN
df = df.replace('?', np.nan)

# Convert string representations to numeric for columns containing NaNs
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')

df.head()

## 2. Basic Dataset Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Missing Value Analysis

In [ ]:
missing_values = df.isnull().sum()
missing_pct = 100 * missing_values / len(df)
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Percentage (%)': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

Only `ca` (4 missing values) and `thal` (2 missing values) contain missing values. These can be imputed during preprocessing (using median for `ca` and mode/most frequent for `thal`).

## 4. Target Class Distribution Analysis
The target has values 0 (absence) and 1, 2, 3, 4 (presence of heart disease with varying severity).
We convert it to a binary target: 0 (absence) vs 1 (presence).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original severity distributions
sns.countplot(x='target', data=df, ax=ax1, hue='target', palette='crest', legend=False)
ax1.set_title("Original Severity Target Classes")
ax1.set_xlabel("Severity Score (0 = No Disease, 1-4 = Disease severity)")

# Binary target mapping
binary_df = df.copy()
binary_df['target'] = binary_df['target'].apply(lambda x: 1 if x > 0 else 0)

sns.countplot(x='target', data=binary_df, ax=ax2, hue='target', palette='Set2', legend=False)
ax2.set_title("Binary Classification Target (Class Balance)")
ax2.set_xticklabels(['No Disease (0)', 'Disease Present (1)'])
ax2.set_xlabel("Target")

plt.tight_layout()
plt.show()

print(f"Class counts:\n{binary_df['target'].value_counts()}")
print(f"Percentage: \n{binary_df['target'].value_counts(normalize=True) * 100}")

The target is relatively balanced: 54% absence (164 patients) vs 46% presence (139 patients), indicating that standard accuracy, recall, and ROC-AUC are all appropriate metrics.

## 5. Numerical Feature Distributions

In [ ]:
num_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f"Distribution of {col}")

# Remove the unused subplot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

## 6. Correlation Analysis
Analyzing relationships between features using Pearson's Correlation Matrix. Categorical variables containing strings must be mapped first to avoid errors.

In [ ]:
# Make target binary
corr_df = df.copy()
corr_df['target'] = corr_df['target'].apply(lambda x: 1 if x > 0 else 0)

# Compute correlation matrix
corr_matrix = corr_df.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={'shrink': .8})
plt.title("Correlation Matrix Heatmap (Binary Target Included)")
plt.show()

### Correlation Insights:
- **Positive Correlation with Target**: `oldpeak` (0.42), `ca` (0.46), `exang` (0.43), `cp` (0.41).
- **Negative Correlation with Target**: `thalach` (-0.42) - meaning patients with lower maximum heart rate are more likely to have heart disease.

## 7. Key Feature Relationships

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Age vs Max Heart Rate achieved split by target
sns.scatterplot(x='age', y='thalach', hue='target', data=corr_df, palette='Set1', alpha=0.8, ax=ax1)
ax1.set_title("Age vs Maximum Heart Rate Achieved")
ax1.set_xlabel("Age (years)")
ax1.set_ylabel("Thalach")

# Cholesterol vs Age split by target
sns.scatterplot(x='age', y='chol', hue='target', data=corr_df, palette='Set1', alpha=0.8, ax=ax2)
ax2.set_title("Age vs Serum Cholesterol")
ax2.set_xlabel("Age (years)")
ax2.set_ylabel("Cholesterol (mg/dl)")

plt.tight_layout()
plt.show()